# TTS bootcamp – full pipeline (Colab / Kaggle, T4 GPU)

1. Runtime → T4 GPU. Add secret `HF_TOKEN` (terms accepted at huggingface.co/ai4bharat/IndicF5).
2. Set `REPO` to where this repository is (clone or Drive). It must contain `kit/` and `submission/`.
3. Run all. If a cell says the length fix was just applied, restart the session and run all again (once).
4. Download `submission/<ROLL>_tts_bootcamp.zip`.

In [ ]:
REPO = "/content/InterIITNLP2"   # where the repository is cloned
BRANCH = "claude/cool-tesla-oqlizq"
ROLL = "ROLLNO"                   # your roll number
import os, subprocess
if not os.path.isdir(f"{REPO}/submission"):
    url = "https://github.com/Mihir269/InterIITNLP2.git"
    # Private repo? Add a Colab secret GH_TOKEN (a GitHub token with read access to the repo).
    try:
        from google.colab import userdata
        tok = userdata.get("GH_TOKEN")
        url = f"https://{tok}@github.com/Mihir269/InterIITNLP2.git"
    except Exception:
        pass
    subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, url, REPO], check=True)
print(os.listdir(REPO))

In [ ]:
import os, subprocess, sys
os.chdir(f"{REPO}/submission")
def sh(cmd):
    print("$", cmd)
    r = subprocess.run(cmd, shell=True, text=True, capture_output=True)
    print(r.stdout[-3000:])
    if r.returncode:
        print(r.stderr[-5000:])
        raise RuntimeError(cmd)
B = f"{REPO}/kit/baseline"
# Colab-friendly install: keep Colab's torch, pin only what IndicF5 needs
sh(f"pip install -q -r {B}/requirements_colab.txt -c {B}/constraints_colab.txt")
sh(f"pip install -q git+https://github.com/ai4bharat/IndicF5.git@fb652a96e508091045890dae5bb99fe511facbcf -c {B}/constraints_colab.txt")
print("Installed. If numpy changed, do Runtime > Restart session once, then run all again.")

In [ ]:
from huggingface_hub import login
try:
    from google.colab import userdata; login(token=userdata.get("HF_TOKEN"))
except Exception:
    try:
        from kaggle_secrets import UserSecretsClient; login(token=UserSecretsClient().get_secret("HF_TOKEN"))
    except Exception as e:
        print(e); login()

In [ ]:
# trigger the one-time length fix (may ask for a restart)
sys.path.insert(0, f"{REPO}/kit")
from baseline.tts import get_engine
get_engine()

In [ ]:
sh(f"KIT={REPO}/kit bash code/run_all.sh")

In [ ]:
sh(f"python code/fill_results.py --roll {ROLL} --machine 'Colab T4'")
sh(f"python code/make_zip.py --roll {ROLL}")
sh(f"python {REPO}/kit/check_format.py {ROLL}_tts_bootcamp.zip")